# CQA evaluator v3: facts + atomic checks + rules
On top of v2:
- **Facts calculated in code** (first/last operator message, how the dialog ended, system identity confirmation, holds) are put into the prompt as trusted facts.
- **Weak criteria are split into small typed fields** (lists of message ids, true/false). Small **rules in code** combine them into the official answer (`derive` key in the checklist). A criterion without `derive` uses the model's score as before, so swapping a criterion still works.
- Voting is done on the **final** (rule-applied) scores. Each result keeps `model_score` and `derived_by`, so the UI can show "decided by rule" or "decided by AI".

Upload `dev.json` and `labels_p1-ru-001-015.json`. The checklist is saved as `checklist_chat-v3.json`; edits to it are picked up on the next run.

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" accelerate

In [ ]:
import torch, json, re, collections, datetime as dt
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
USE_4BIT = False   # 4B in fp16 (~8 GB) fits a T4. Set True only for bigger models (e.g. Qwen3-8B on T4).

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
kw = dict(device_map="auto")
if USE_4BIT:
    kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                   bnb_4bit_compute_dtype=torch.float16)
else:
    kw["dtype"] = torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)

In [ ]:
# ---------- CHECKLIST (single source of truth) ----------
# check.fields: typed helper fields the model fills BEFORE explanation/score.
# derive: name of a rule in RULES (cell below) that turns fields + facts into the final score. Optional.
F = lambda typ, instr, ex: {"type": typ, "instruction": instr, "example": ex}
DEFAULT_CHECKLIST = {
 "checklist_id": "chat-v3",
 "criteria": [
  {"id": 1, "title": "Установление контакта", "type": "binary", "weight": 10, "na_allowed": False,
   "rule": "Да — в ПЕРВОЙ реплике оператора (см. ФАКТЫ) есть приветствие на языке клиента И имя оператора. Приветствие бота не считается. «Привет» — неформально, не считается."},
  {"id": 2, "title": "Выявление потребности", "type": "binary", "weight": 15, "na_allowed": True,
   "rule": "N/A — на момент подключения оператора данных для ответа хватало (учитывай всё, что клиент написал боту). Да — данных не хватало и оператор уточнил недостающее до ответа. Нет — не уточнил нужное или переспросил уже известное.",
   "na_condition": "Данных для ответа хватало с самого начала.",
   "check": {"fields": {
      "missing_info": F("list[str]", "Чего НЕ хватало оператору для ответа в момент подключения. Только то, без чего ответить было нельзя. Если оператор мог ответить сразу — []", []),
      "operator_clarified": F("bool", "Оператор задал уточняющий вопрос о недостающем до ответа", False),
      "reasked_known": F("bool", "Оператор спросил то, что клиент уже сообщил (в т.ч. боту)", False)}},
   "derive": "need"},
  {"id": 3, "title": "Идентификация", "type": "binary", "weight": 20, "na_allowed": True,
   "rule": "Да — личность подтверждена ДО того, как оператор сообщил сведения о счёте клиента или выполнил действие по счёту. Нет — сообщил/выполнил без подтверждения. N/A — оператор не сообщал сведений о конкретном счёте и не выполнял действий по нему.",
   "na_condition": "Запрос общий, сведения о счёте не раскрывались.",
   "check": {"fields": {
      "account_info_msgs": F("list[int]", "Номера реплик ОПЕРАТОРА, где он сообщает сведения о КОНКРЕТНОМ счёте клиента: ограничения, баланс, позиции, статус платежа, персональные данные. Общие инструкции и тарифы — не сведения о счёте", []),
      "account_action_msgs": F("list[int]", "Номера реплик ОПЕРАТОРА, где он выполняет или запускает действие по счёту клиента (проверка счёта, передача обращения по счёту, изменение данных)", []),
      "verbal_verification_msg": F("int|null", "Только для звонка: номер реплики, где оператор подтвердил личность по названным клиентом данным. Иначе null", None)}},
   "derive": "identification"},
  {"id": 4, "title": "Корректная и полная консультация", "type": "scale", "weight": 20, "na_allowed": True,
   "scale": {"20": "Все существенные вопросы закрыты или дан корректный следующий шаг.",
             "15": "Основной вопрос решён, пропущена второстепенная деталь.",
             "10": "Часть запроса закрыта, существенный вопрос остался открытым.",
             "5": "Есть общая подсказка, но клиенту всё ещё непонятно, как действовать.",
             "0": "Ответа по существу нет или ключевая рекомендация ошибочна."},
   "na_condition": "Чат закончился до того, как клиент сформулировал запрос.",
   "check": {"fields": {
      "client_questions": F("list[obj]", "ВСЕ вопросы клиента (включая заданные боту до оператора). status: «закрыт» | «следующий шаг» (дан корректный следующий шаг или передано на проверку) | «открыт-главный» | «открыт-второстепенный». repeat_of — номер более раннего вопроса, который клиент повторяет или переформулирует, иначе null",
                            [{"msg": 0, "question": "где тариф", "status": "закрыт", "repeat_of": None}]),
      "wrong_info_msgs": F("list[int]", "Реплики оператора с явно ошибочной информацией или рекомендацией", [])}},
   "derive": "consultation"},
  {"id": 5, "title": "Эмпатия и клиентоориентированность", "type": "binary", "weight": 15, "na_allowed": False,
   "rule": "Да — оператор вежлив и, если клиент выразил недовольство, возражение или неудобство, отреагировал на него по существу. Нет — невежлив или проигнорировал недовольство/возражение клиента (ответил мимо, отмахнулся). Извинения не обязательны."},
  {"id": 6, "title": "Управление диалогом, скорость и грамотность", "type": "scale", "weight": 15, "na_allowed": False,
   "scale": {"10": "Ответы ясные и связные. Если было явное ожидание, оператор предупредил.",
             "7": "Один заметный сбой, но клиент может понять ответ и продолжить.",
             "4": "Клиенту приходится переспрашивать из-за формулировок или порядка ответов.",
             "0": "Из-за бессвязных или противоречивых реплик ход диалога непонятен."},
   "rule": "Полнота ответа — это критерий 4, здесь не оценивается. Порог первой реакции 3 минуты не оценивается.",
   "voice_rule": "Для звонка оценивай логичность разговора, понятность объяснений, отсутствие неуместных выражений, корректность формулировок и управление паузами/удержанием (см. ФАКТЫ). Искажения распознавания речи — не вина оператора.",
   "check": {"fields": {
      "unclear_operator_msgs": F("list[int]", "Реплики оператора, которые непонятны, противоречивы, не по заданному вопросу или задают лишний вопрос", [])}},
   "derive": "dialog_management"},
  {"id": 7, "title": "Завершение", "type": "binary", "weight": 5, "na_allowed": True,
   "rule": "Да — в конце оператор подвёл итог (повторил результат или следующий шаг) ИЛИ предложил дальнейшую помощь, И попрощался. Нет — только прощание или закрыл без прощания. N/A — клиент ушёл / связь оборвалась раньше (см. ФАКТЫ).",
   "na_condition": "Клиент ушёл или связь оборвалась до того, как оператор мог завершить.",
   "check": {"fields": {
      "has_farewell": F("bool", "В последних репликах оператора есть прощание (до свидания, хорошего дня, сау болыңыз, goodbye, have a good day…)", True),
      "has_summary_or_offer": F("bool", "В последних репликах оператора есть итог (повтор результата или следующего шага, напр. «Вопрос передан», «начните со сверки дат») ИЛИ предложение помощи («если вопрос останется, напишите»). Слова «Пожалуйста», «Отлично», «Хорошо», «Спасибо за обращение» сами по себе НЕ итог", False)}},
   "derive": "closing"},
  {"id": 8, "title": "Грубые ошибки", "type": "binary", "weight": 0, "na_allowed": False, "critical": True,
   "rule": "Да — оскорбление клиента, ИЛИ предложение платной/личной услуги в обход компании (достаточно предложения), ИЛИ раскрытие конфиденциальной информации. Невежливый тон без оскорбления — не грубая ошибка.",
   "check": {"fields": {
      "insult_msgs": F("list[int]", "Реплики оператора с оскорблением клиента", []),
      "private_offer_msgs": F("list[int]", "Реплики оператора с предложением перевести деньги лично ему, платной/частной консультации, связи по личному номеру", [])}},
   "derive": "gross_errors"}
 ]}

CHECKLIST_PATH = Path(f"checklist_{DEFAULT_CHECKLIST['checklist_id']}.json")   # versioned: an old v2 file is never picked up by mistake
def load_checklist(path=CHECKLIST_PATH):
    if not Path(path).exists():
        Path(path).write_text(json.dumps(DEFAULT_CHECKLIST, ensure_ascii=False, indent=2), encoding="utf-8")
    return json.loads(Path(path).read_text(encoding="utf-8"))

def allowed_values(c):
    vals = {int(v) for v in c["scale"]} if c["type"] == "scale" else {"Да", "Нет"}
    if c.get("na_allowed"): vals.add("N/A")
    return vals

def check_fields(c): return c.get("check", {}).get("fields", {})

def render_checklist(ck, channel):
    out = []
    for c in ck["criteria"]:
        vals = sorted([v for v in allowed_values(c) if v != "N/A"], reverse=True) if c["type"] == "scale" else ["Да", "Нет"]
        out.append(f"КРИТЕРИЙ {c['id']} — {c['title'].upper()}  (допустимо: {', '.join(map(str, vals))}"
                   + (", N/A" if c.get("na_allowed") else "; N/A запрещено") + ")")
        if c.get("rule"): out.append(c["rule"])
        for k, v in sorted(c.get("scale", {}).items(), key=lambda kv: -int(kv[0])): out.append(f"  {k}: {v}")
        if c.get("na_condition"): out.append("N/A — " + c["na_condition"])
        if channel == "voice call" and c.get("voice_rule"): out.append("ДЛЯ ЗВОНКА: " + c["voice_rule"])
        for name, f in check_fields(c).items(): out.append(f"  поле {name} ({f['type']}): {f['instruction']}")
        out.append("")
    return "\n".join(out)

checklist = load_checklist()
print(render_checklist(checklist, "agent chat")[:1200])

In [ ]:
# ---------- DATASET from organizers' API JSON, with time gaps computed in code ----------
def _sec(ts):
    if "T" in ts: return dt.datetime.fromisoformat(ts).timestamp()
    m, s = ts.split(":"); return int(m) * 60 + int(s)

def format_dialogue(rec):
    # show waits only before OPERATOR replies: >=60 s in chats, >=10 s in calls
    gap_threshold = 10 if rec["channel"] == "voice call" else 60
    lines, prev = [], None
    for i, m in enumerate(rec["transcript"]):
        t = _sec(m["timestamp"]); shown = m["timestamp"][11:19] if "T" in m["timestamp"] else m["timestamp"]
        gap = f" (пауза {int(t - prev)} с)" if prev is not None and m["speaker"] == "operator" and t - prev >= gap_threshold else ""
        lines.append(f"[{i}] [{shown}]{gap} {m['speaker']}: {m['text']}")
        prev = t
    return "\n".join(lines)

raw = json.load(open("dev.json", encoding="utf-8"))
records = raw["data"]["records"] if "data" in raw else raw
DATA = {r["conversation_id"]: {"conversation_id": r["conversation_id"], "channel": r["channel"],
        "language": r["language"], "n_messages": len(r["transcript"]), "dialogue_text": format_dialogue(r)}
        for r in records}
print(len(DATA)); print(DATA["p1-ru-003"]["dialogue_text"])

In [ ]:
# ---------- FACTS computed in code (trusted, given to the model and to the rules) ----------
END_CLIENT = ["клиент покинул", "клиент завершил", "соединение с клиентом прервано", "байланыс үзілді", "чаттан шықты",
              "customer left", "connection to customer lost", "client left", "customer ended"]
END_OPERATOR = ["оператор закрыл чат", "оператор чатты жапты", "agent closed the chat"]
VERIFIED = ["личность клиента подтверждена", "жеке басы расталды", "identity verified"]
HOLD_ON, HOLD_OFF = ["находится на удержании"], ["больше не находится на удержании"]
has = lambda text, keys: any(k in text.lower() for k in keys)

def compute_facts(rec):
    t = rec["transcript"]
    op = [i for i, m in enumerate(t) if m["speaker"] == "operator"]
    f = {"first_operator_msg": op[0] if op else None, "last_operator_msg": op[-1] if op else None,
         "verified_msg": next((i for i, m in enumerate(t) if m["speaker"] == "system" and has(m["text"], VERIFIED)), None),
         "end_msg": None, "end_by": None, "holds": []}
    for i, m in enumerate(t):
        if m["speaker"] != "system": continue
        if has(m["text"], END_CLIENT): f.update(end_msg=i, end_by="client")
        elif has(m["text"], END_OPERATOR): f.update(end_msg=i, end_by="operator")
        if has(m["text"], HOLD_OFF) and f["holds"] and f["holds"][-1][1] is None:
            f["holds"][-1][1] = i; f["holds"][-1][2] = int(_sec(m["timestamp"]) - _sec(t[f["holds"][-1][0]]["timestamp"]))
        elif has(m["text"], HOLD_ON): f["holds"].append([i, None, None])
    return f

def facts_text(rec, f):
    t = rec["transcript"]; L = []
    L.append(f"- Первая реплика оператора: [{f['first_operator_msg']}]" if f["first_operator_msg"] is not None else "- Оператор не писал.")
    if f["last_operator_msg"] is not None:
        L.append(f"- Последняя реплика оператора: [{f['last_operator_msg']}] «{t[f['last_operator_msg']]['text']}»")
    if f["end_msg"] is not None:
        who = "клиент ушёл / связь оборвалась" if f["end_by"] == "client" else "оператор закрыл чат"
        L.append(f"- Завершение: [{f['end_msg']}] «{t[f['end_msg']]['text']}» — {who}.")
    else: L.append("- Системного сообщения о завершении нет.")
    L.append(f"- Системное подтверждение личности: [{f['verified_msg']}]." if f["verified_msg"] is not None
             else "- Системного подтверждения личности в диалоге НЕТ.")
    for a, b, s in f["holds"]: L.append(f"- Удержание на линии: [{a}]→[{b}], {s} с.")
    return "\n".join(L)

for cid in DATA:
    rec = next(r for r in records if r["conversation_id"] == cid)
    DATA[cid]["facts"] = compute_facts(rec); DATA[cid]["facts_text"] = facts_text(rec, DATA[cid]["facts"])
print(DATA["p1-ru-009"]["facts_text"]); print(DATA["p4-ru-002"]["facts_text"])

In [ ]:
SYSTEM_PROMPT = """Ты — аудитор качества поддержки Freedom Broker. Оцени работу ОПЕРАТОРА строго по чек-листу.
Правила:
1. Оценивай только реплики operator. client, bot и system — контекст. Действия бота не засчитываются оператору.
2. Блок ФАКТЫ вычислен программой — считай его верным.
3. Используй только то, что есть в диалоге. Каждое утверждение подкрепляй номером реплики. Не придумывай недостатки: если замечание нельзя привязать к реплике — его нет.
4. «Нет» — действие требовалось и не выполнено. N/A — только где разрешено и выполнено условие неприменимости.
5. Не путай критерии: 4 — правильность и полнота решения; 6 — ясность и порядок общения; 5 — вежливость и реакция на недовольство; 7 — итог/предложение помощи + прощание.
6. Сначала заполни поля критерия, потом explanation (не более 20 слов, на русском), потом evidence_ids, в конце score. Итоговый балл не считай.
7. Поля заполняй в указанном типе: list[int] — список чисел, bool — true/false, list[obj] — список объектов. Не оборачивай их в строки.
8. Верни только JSON, без Markdown."""

def build_prompt(row, ck):
    ids = [c["id"] for c in ck["criteria"]]; example = []
    for c in ck["criteria"]:
        e = {"id": c["id"]}
        for name, f in check_fields(c).items(): e[name] = f["example"]
        e.update({"explanation": "...", "evidence_ids": [10], "score": "Да" if c["type"] == "binary" else max(int(k) for k in c["scale"])})
        example.append(e)
    user = f"""ЧЕК-ЛИСТ:
{render_checklist(ck, row['channel'])}
ФАКТЫ:
{row['facts_text']}

ДИАЛОГ (conversation_id: {row['conversation_id']}, канал: {row['channel']}, язык: {row['language']}; номера реплик с 0):
{row['dialogue_text']}

Верни JSON (значения в примере — только формат, не ответы): {{"conversation_id": "{row['conversation_id']}", "criteria": {json.dumps(example, ensure_ascii=False)}}}
Ровно {len(ids)} объектов с id {ids}."""
    return [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]

In [ ]:
# ---------- generation, parsing, normalization ----------
def generate(messages, max_new_tokens=1500, sample=False):
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    gen.update(dict(do_sample=True, temperature=0.3, top_p=0.9) if sample else dict(do_sample=False))
    with torch.inference_mode():
        out = model.generate(**inputs, **gen)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

def extract_json(text):
    text = re.sub(r"^```(json)?|```$", "", text.strip(), flags=re.M)
    s, e = text.find("{"), text.rfind("}")
    if s < 0 or e < 0: raise ValueError("no JSON object in output")
    return json.loads(re.sub(r",\s*([}\]])", r"\1", text[s:e + 1]))

_MAP = {"да": "Да", "yes": "Да", "нет": "Нет", "no": "Нет", "n/a": "N/A", "na": "N/A", "н/д": "N/A"}
def normalize_score(v, c):
    if isinstance(v, str):
        s = v.strip()
        v = int(float(s)) if re.fullmatch(r"-?\d+(\.0+)?", s) else _MAP.get(s.lower(), s)
    if isinstance(v, float) and v.is_integer(): v = int(v)
    allowed = allowed_values(c)
    if v not in allowed and c["type"] == "scale" and isinstance(v, int):
        v = min((a for a in allowed if a != "N/A"), key=lambda a: abs(a - v))
    if v not in allowed: raise ValueError(f"criterion {c['id']}: invalid score {v!r}, allowed {allowed}")
    return v

def _ids(v, n):
    if isinstance(v, str):
        try: v = json.loads(v)
        except Exception: v = re.findall(r"\d+", v)
    if not isinstance(v, list): v = [v] if v is not None else []
    out = []
    for i in v:
        if str(i).strip().lstrip("-").isdigit() and 0 <= int(i) < n: out.append(int(i))
    return out

def _bool(v):
    if isinstance(v, str): return v.strip().lower() in ("true", "да", "yes", "1")
    return bool(v)

def coerce_field(v, typ, n):
    if typ == "bool": return _bool(v)
    if typ == "list[int]": return _ids(v, n)
    if typ == "int|null":
        ids = _ids(v, n); return ids[0] if ids else None
    if typ in ("list[str]", "list[obj]"):
        if isinstance(v, str):
            try: v = json.loads(v)
            except Exception: v = [v] if v.strip() and v.strip() not in ("[]", "-", "нет") else []
        return v if isinstance(v, list) else []
    return v

def parse_and_validate(text, row, ck):
    res = extract_json(text); n = row["n_messages"]
    by_id = {int(x["id"]): x for x in res.get("criteria", []) if str(x.get("id", "")).isdigit()}
    crit = []
    for c in ck["criteria"]:
        if c["id"] not in by_id: raise ValueError(f"missing criterion {c['id']}")
        x = by_id[c["id"]]; x["id"] = c["id"]
        x["score"] = normalize_score(x.get("score"), c)
        x["evidence_ids"] = _ids(x.get("evidence_ids", []), n)
        for name, f in check_fields(c).items(): x[name] = coerce_field(x.get(name, f["example"]), f["type"], n)
        crit.append(x)
    return crit

In [ ]:
# ---------- RULES: fields + facts -> final score. Return None to keep the model's score. ----------
def r_need(x, run, row, f):
    if x["reasked_known"]: return "Нет"
    if not x["missing_info"]: return "N/A"
    return "Да" if x["operator_clarified"] else "Нет"

def _unverified(x, f):
    msgs = sorted(set(x["account_info_msgs"]) | set(x["account_action_msgs"]))
    v = [i for i in (f["verified_msg"], x.get("verbal_verification_msg")) if i is not None]
    v = min(v) if v else None
    return msgs, [m for m in msgs if v is None or m < v]

def r_identification(x, run, row, f):
    msgs, bad = _unverified(x, f)
    if not msgs: return "N/A"
    x["evidence_ids"] = sorted(set(x["evidence_ids"]) | set(bad or msgs))
    return "Нет" if bad else "Да"

def _questions(x):
    qs = [q for q in x["client_questions"] if isinstance(q, dict)]
    repeated = {q.get("repeat_of") for q in qs if isinstance(q.get("repeat_of"), int)}
    return qs, [q for q in qs if q.get("msg") not in repeated]          # latest version of each question

def r_consultation(x, run, row, f):
    if x["score"] == "N/A": return None
    qs, live = _questions(x)
    if not qs: return None
    st = [str(q.get("status", "")).lower() for q in live]
    if x["wrong_info_msgs"] and x["score"] in (0, 5): return x["score"]
    if any("главн" in s for s in st): return x["score"] if x["score"] in (0, 5, 10) else 10
    if any("второстеп" in s for s in st): return 15
    if any(s.startswith("открыт") for s in st): return x["score"] if x["score"] in (0, 5, 10) else 10
    return 20

def r_dialog_management(x, run, row, f):
    if row["channel"] == "voice call" or x["score"] == 0: return None   # calls: model judges
    c4 = next((r for r in run if "client_questions" in r), None)
    fo = f["first_operator_msg"] or 0     # only re-asks AFTER the operator joined count (repeating a bot question doesn't)
    repeats = [q for q in (c4["client_questions"] if c4 else []) if isinstance(q, dict)
               and isinstance(q.get("repeat_of"), int) and isinstance(q.get("msg"), int) and q["msg"] > fo]
    if repeats:
        x["evidence_ids"] = sorted(set(x["evidence_ids"]) | {q["msg"] for q in repeats if isinstance(q.get("msg"), int)})
        return 4
    return 7 if x["unclear_operator_msgs"] else 10

def r_closing(x, run, row, f):
    if f["last_operator_msg"] is None: return "N/A"
    if f["end_by"] == "client" and not x["has_farewell"]: return "N/A"
    x["evidence_ids"] = sorted(set(x["evidence_ids"]) | {f["last_operator_msg"]})
    return "Да" if x["has_farewell"] and x["has_summary_or_offer"] else "Нет"

def r_gross_errors(x, run, row, f):
    c3 = next((r for r in run if "account_info_msgs" in r), None)
    leaked = _unverified(c3, f)[1] if c3 else []
    leaked = [m for m in leaked if c3 and m in c3["account_info_msgs"]]   # disclosure, not just an action
    hits = sorted(set(x["insult_msgs"]) | set(x["private_offer_msgs"]) | set(leaked))
    if hits: x["evidence_ids"] = sorted(set(x["evidence_ids"]) | set(hits))
    return "Да" if hits else "Нет"

RULES = {"need": r_need, "identification": r_identification, "consultation": r_consultation,
         "dialog_management": r_dialog_management, "closing": r_closing, "gross_errors": r_gross_errors}

def apply_rules(run, row, ck):
    for c, x in zip(ck["criteria"], run):
        x["model_score"] = x["score"]; x["derived_by"] = "model"
        rule = RULES.get(c.get("derive"))
        if not rule: continue
        try:
            s = rule(x, run, row, row["facts"])
            if s is not None:
                x["score"] = normalize_score(s, c); x["derived_by"] = "rule:" + c["derive"]
        except Exception as e:
            x["rule_error"] = str(e)
    return run

In [ ]:
# ---------- scoring from checklist (no hardcoded ids/weights) ----------
def calculate_score(criteria, ck):
    cfg = {c["id"]: c for c in ck["criteria"]}
    earned = applicable = 0; severe = False
    for x in criteria:
        c = cfg[x["id"]]; s = x["score"]
        if c.get("critical"):
            severe = severe or s == "Да"; continue
        if s == "N/A" or c["weight"] == 0: continue
        applicable += c["weight"]
        frac = s / max(int(k) for k in c["scale"]) if c["type"] == "scale" else (1 if s == "Да" else 0)
        x["points"] = round(c["weight"] * frac, 2); earned += c["weight"] * frac
    return {"total_score": round(earned / applicable * 100, 2) if applicable else None, "severe_violation": severe}

def evaluate_dialogue(row, ck, n_runs=1, max_new_tokens=1500, retries=1):
    """n_runs=1: greedy. n_runs=3: sampled runs + majority vote on FINAL (rule-applied) scores."""
    msgs = build_prompt(row, ck); runs = []; last_err = None
    for r in range(n_runs):
        for attempt in range(retries + 1):
            try:
                runs.append(apply_rules(parse_and_validate(generate(msgs, max_new_tokens, sample=n_runs > 1), row, ck), row, ck)); break
            except Exception as e: last_err = e
        else: print(f"{row['conversation_id']} run {r} failed: {last_err}")
    if not runs: raise RuntimeError(f"all runs failed: {last_err}")
    final = []
    for i, c in enumerate(ck["criteria"]):
        votes = collections.Counter(str(run[i]["score"]) for run in runs)
        top, n = votes.most_common(1)[0]
        final.append(dict(next(run[i] for run in runs if str(run[i]["score"]) == top), agreement=round(n / len(runs), 2)))
    out = {"conversation_id": row["conversation_id"], "checklist_id": ck["checklist_id"],
           "language": row["language"], "channel": row["channel"], "facts": row["facts"], "criteria": final}
    out.update(calculate_score(out["criteria"], ck))
    out["needs_review"] = any(x["agreement"] < 1 for x in final)
    return out

In [ ]:
# ---------- quick single-dialog test (same code path as batch) ----------
r = evaluate_dialogue(DATA["p1-ru-003"], checklist)
print(json.dumps(r, ensure_ascii=False, indent=2))

In [ ]:
# ---------- batch ----------
IDS = [f"p1-ru-{i:03d}" for i in range(1, 16)]     # labeled subset; use list(DATA) for all 65
N_RUNS = 1                                          # try 3 once the single-run version is good
RUN_NAME = "v3_facts_rules"

results, errors = [], []
from tqdm.auto import tqdm
for cid in tqdm(IDS):
    try: results.append(evaluate_dialogue(DATA[cid], checklist, n_runs=N_RUNS))
    except Exception as e: errors.append({"conversation_id": cid, "error": str(e)}); print(cid, e)
Path(f"results_{RUN_NAME}.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"ok {len(results)}/{len(IDS)}, errors {len(errors)}")

In [ ]:
# ---------- accuracy vs labels (final score vs. model's own score, to see what the rules add) ----------
def accuracy(results, labels_path="labels_p1-ru-001-015.json", show_errors=True):
    lab = json.load(open(labels_path, encoding="utf-8"))
    lab = {x["conversation_id"]: x for x in lab.get("labels", lab)}
    hit, hit_m, tot, mae, errs = collections.Counter(), collections.Counter(), collections.Counter(), [], []
    for r in results:
        g = lab.get(r["conversation_id"])
        if not g: continue
        gold = {c["id"]: c for c in g["criteria"]}
        for c in r["criteria"]:
            if c["id"] not in gold: continue
            gs = str(gold[c["id"]]["score"]); tot[c["id"]] += 1
            hit_m[c["id"]] += str(c.get("model_score", c["score"])) == gs
            if str(c["score"]) == gs: hit[c["id"]] += 1
            else: errs.append(f"{r['conversation_id']} c{c['id']}: final {c['score']} (model {c.get('model_score')}, {c.get('derived_by')})"
                              f" vs label {gs} ({gold[c['id']].get('confidence','high')}) — {c.get('explanation','')}")
        if r["total_score"] is not None and g["total_score"] is not None: mae.append(abs(r["total_score"] - g["total_score"]))
    print("crit  final  model-only")
    for k in sorted(tot): print(f"c{k}:  {hit[k]/tot[k]:>5.0%}  {hit_m[k]/tot[k]:>6.0%}   ({hit[k]}/{tot[k]})")
    T = sum(tot.values())
    print(f"OVERALL final {sum(hit.values())/T:.0%} | model-only {sum(hit_m.values())/T:.0%} | total-score MAE {sum(mae)/max(1,len(mae)):.2f}")
    if show_errors: print("\n".join(errs))

accuracy(results)

In [ ]:
# ---------- debugging helper: see the raw model output + fields for one dialog ----------
# raw = generate(build_prompt(DATA["p1-ru-009"], checklist)); print(raw)
# r = [x for x in results if x["conversation_id"] == "p1-ru-009"][0]
# for c in r["criteria"]: print(c["id"], c["model_score"], "->", c["score"], c["derived_by"], {k: v for k, v in c.items() if k not in ("id","score","model_score","derived_by","explanation","evidence_ids","agreement","points")})